# Data Access

> Functionality to access Euclid calibrated data frames for specific targets and observation IDs.

In [ ]:
# | default_exp euclid.data_access

In [ ]:
from astropy import table
from astropy.table import Table
from astroquery.utils.tap.core import TapPlus
from getpass import getpass

import os

import numpy as np

In [ ]:
# |export

class DataAccess:
    """Provides access to Euclid data."""
    
    def __init__(self,
                 esa_username=None,  # ESA account username (prompts if not supplied)
                 esa_password=None,  # ESA account password (prompts if not supplied)
                 esac_server_url="https://easotf.esac.esa.int",  # ESA server (default is on-the-fly)
                 dry_run=False,  # if True, do not actually download files
                ):
        """Create an object for accessing data and log in to the ESA server."""
        if esa_username is None:
            esa_username = getpass(prompt="ESA User:")
        if esa_password is None:
            esa_password = getpass(prompt="ESA Password:")
        self.dry_run = dry_run
        self.tap = TapPlus(url=f"{esac_server_url}/tap-server", tap_context="tap")
        self.data_tap = TapPlus(url=f"{esac_server_url}/sas-dd", data_context="data")
        self.tap.login(user=esa_username, password=esa_password)
        self.data_tap.login(user=esa_username, password=esa_password)

    def find_observations_for_target(self,
                                     ra,  # RA of the target, in decimal degrees
                                     dec,  # Dec of the target, in decimal degrees
                                     radius=1/60,  # radius of the target, in decimal degrees
                                    ):  # returns a list of observation_ids
        """Obtain a list of obs_ids for observations that entirely contain the specified target region."""
        query = f"""SELECT observation_stack.observation_id
                    FROM sedm.observation_stack
                    AS observation_stack
                    WHERE (product_type like '%Stacked%')
                    AND ((observation_stack.fov IS NOT NULL AND CONTAINS(CIRCLE('ICRS',{ra},{dec},{radius}),observation_stack.fov)=1))
                    ORDER BY observation_id ASC"""
        job = self.tap.launch_job(query)
        results = job.get_results()
        obs_ids = np.unique(list(results["observation_id"])).astype(int)
        return obs_ids

    def get_calibrated_files_for_observation(self,
                                             obs_id,  # observation_id for which to find files
                                             instrument=None,  # None, NISP or VIS
                                             filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
                                            ):  # returns a table of file information
        """Obtain file information for obs_id, optionally restricted by instrument or filter."""
        instrument_condition = f"AND instrument_name = '{instrument}'" if instrument is not None else ""
        filter_condition = f"AND filter_name = '{filter}'" if filter is not None else ""
        query = f"""SELECT observation_stack.observation_id, observation_stack.file_name,
                    observation_stack.instrument_name, observation_stack.filter_name, observation_stack.duration
                    FROM sedm.calibrated_frame
                    AS observation_stack
                    WHERE (product_type like '%Calibrated%')
                    {instrument_condition}
                    {filter_condition}
                    AND observation_id = '{obs_id}'"""
        job = self.tap.launch_job(query)
        file_info = job.get_results()
        return file_info

    def download_files(self,
                       filenames,  #  list of filenames or Table containing "file_name" column
                       outpath="./",  # the folder in which to save the downloaded files
                       verbose=True,  # print information to the screen
                      ):
        """Download multiple Euclid filenames to outpath."""
        if isinstance(filenames, Table):
            filenames = filenames["file_name"]
        if verbose:
            print(f"Downloading {len(filenames)} files to {outpath}")
        for fn in filenames:
            if verbose:
                print(f"Downloading {fn}")
            self.download_file(fn, outpath)

    def download_file(self,
                      filename,  # the filename to download
                      outpath="./",  # the folder in which to save the downloaded files
                     ):
        """Download Euclid filename to outpath."""
        params_dict = dict(RETRIEVAL_TYPE="FILE", RELEASE="sedm", RETRIEVAL_ACCESS="DIRECT")
        params_dict.update(FILE_NAME=filename)
        outpath = os.path.expanduser(outpath)
        outfn = os.path.join(outpath, filename)
        if not self.dry_run:
            self.data_tap.load_data(params_dict=params_dict, output_file=outfn)

    def download_calibrated_files_for_observation(self,
                                                  obs_id,
                                                  outpath="./",  # the folder in which to save the downloaded files
                                                  instrument=None,  # None, NISP or VIS
                                                  filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
                                                  verbose=True,  # print information to the screen
                                                 ):  #  returns a table of file information
        """Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter."""
        file_info = self.get_calibrated_files_for_observation(obs_id, instrument=instrument, filter=filter)
        self.download_files(file_info, outpath=outpath, verbose=verbose)
        return file_info

    def download_calibrated_files_for_target(self,
                                             ra,  # RA of the target, in decimal degrees
                                             dec,  # Dec of the target, in decimal degrees
                                             radius=1/60,  # radius of the target, in decimal degrees
                                             outpath="./",  # the folder in which to save the downloaded files
                                             instrument=None,  # None, NISP or VIS
                                             filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
                                             verbose=True,  # print information to the screen
                                            ):  #  returns a table of file information
        """Download all calibrated files for Euclid observations covering a target, optionally restricted by instrument or filter."""
        file_info = []
        obs_ids = self.find_observations_for_target(ra, dec, radius)
        for obs_id in obs_ids:
            if verbose:
                print(f"Downloading files for observation id {obs_id}")
            obs_file_info = self.download_calibrated_files_for_observation(obs_id, outpath=outpath, instrument=instrument,
                                                                           filter=filter, verbose=verbose)
            file_info.append(obs_file_info)
        return table.vstack(file_info)

The following demonstrates use of the DataAccess class. This would normally first be imported using `from nicl.euclid.data_access import DataAccess`.

To actually download files, remove `dry_run=True`.

In [ ]:
da = DataAccess(dry_run=True)

ESA User: ········
ESA Password: ········


INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


In [ ]:
da.find_observations_for_target(ra=265.799, dec=64.663)

array([2070, 2109, 2228,  524])

In [ ]:
da.get_calibrated_files_for_observation(2070, instrument="NISP", filter="NIR_H")

observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2070,EUC_NIR_W-CAL-IMAGE_H-2070-0_20240706T082906.487566Z.fits,NISP,NIR_H,87.2448
2070,EUC_NIR_W-CAL-IMAGE_H-2070-2_20240706T082906.475051Z.fits,NISP,NIR_H,87.2448
2070,EUC_NIR_W-CAL-IMAGE_H-2070-3_20240706T082906.313876Z.fits,NISP,NIR_H,87.2448
2070,EUC_NIR_W-CAL-IMAGE_H-2070-1_20240706T082906.407839Z.fits,NISP,NIR_H,87.2448


In [ ]:
da.download_calibrated_files_for_observation(2070, instrument="NISP", filter="NIR_H", outpath="~/data/euclid/test/")

observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2070,EUC_NIR_W-CAL-IMAGE_H-2070-0_20240706T082906.487566Z.fits,NISP,NIR_H,87.2448
2070,EUC_NIR_W-CAL-IMAGE_H-2070-3_20240706T082906.313876Z.fits,NISP,NIR_H,87.2448
2070,EUC_NIR_W-CAL-IMAGE_H-2070-2_20240706T082906.475051Z.fits,NISP,NIR_H,87.2448
2070,EUC_NIR_W-CAL-IMAGE_H-2070-1_20240706T082906.407839Z.fits,NISP,NIR_H,87.2448


In [ ]:
da.download_calibrated_files_for_target(ra=265.799, dec=64.663, instrument="NISP", filter="NIR_H")

observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2070,EUC_NIR_W-CAL-IMAGE_H-2070-0_20240706T082906.487566Z.fits,NISP,NIR_H,87.2448
2070,EUC_NIR_W-CAL-IMAGE_H-2070-3_20240706T082906.313876Z.fits,NISP,NIR_H,87.2448
2070,EUC_NIR_W-CAL-IMAGE_H-2070-1_20240706T082906.407839Z.fits,NISP,NIR_H,87.2448
2070,EUC_NIR_W-CAL-IMAGE_H-2070-2_20240706T082906.475051Z.fits,NISP,NIR_H,87.2448
2109,EUC_NIR_W-CAL-IMAGE_H-2109-3_20240706T111043.619656Z.fits,NISP,NIR_H,87.2448
2109,EUC_NIR_W-CAL-IMAGE_H-2109-0_20240706T111043.485402Z.fits,NISP,NIR_H,87.2448
2109,EUC_NIR_W-CAL-IMAGE_H-2109-1_20240706T111043.990104Z.fits,NISP,NIR_H,87.2448
2109,EUC_NIR_W-CAL-IMAGE_H-2109-2_20240706T111043.336811Z.fits,NISP,NIR_H,87.2448
2228,EUC_NIR_W-CAL-IMAGE_H-2228-0_20240716T145836.349288Z.fits,NISP,NIR_H,87.2448
